# レッスン 09 - メタ認知デザインパターン


## セットアップ

このノートブックは、Microsoft Agent Framework を使用して Metacognition デザインパターンを示します。

**前提条件：**
- 環境変数で構成された Azure OpenAI デプロイメント
- Azure CLI にログインしていること（`az login`）


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## メタ認知とは何か？

メタ認知は**思考について考えること**です。AIエージェントの文脈では、これは次のことができるエージェントを構築することを意味します:

- **自己反省する** 自分の出力と推論プロセスについて
- **エラーを検出する** 静かに失敗するのではなく、適切に回復する
- **評価する** 応答が完全で有益かどうか
- **適応する** 最初のアプローチがうまくいかないときに戦略を変える (例：バックアップシステムにフォールバックする)

メタ認知型エージェントは単に質問に答えるだけではなく — 自分のパフォーマンスを監視し、その場で調整します。


## プライマリとバックアップツール

一般的なメタ認知のパターンは**フォールバック戦略**です。エージェントはまずプライマリツールを試し; それが失敗した場合（例: 404 エラー）、エージェントは失敗を認識して透明性をもってバックアップツールに切り替えます。

これは、主要なサービスが利用できない場合があり、エージェントが代替の経路を選択する前に問題を自己診断しなければならない現実世界のシステムを反映しています。

以下では、2つのフライト検索ツールを定義します:
- **Primary** — パリ、東京、バルセロナをカバーします
- **Backup** — ベルリン、シドニー、ニューヨーク市をカバーします


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## エラー回復を備えた自己反省型エージェント

以下のエージェントは、まず主要な飛行システムを試し、故障を認識した場合は透明性を持ってバックアップシステムにフォールバックするよう指示されています。各応答の後に、自分の回答がユーザーの質問に完全に回答しているかどうかを簡潔に自己評価します。


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## 自己評価パターン

メタ認知の別の側面は **自己評価** です: 別のエージェント（または同じエージェントが二回目の処理で）が、応答の完全性、正確性、および有用性を評価します。

以下では、旅行エージェントの応答を3つの次元でスコア付けする`ResponseEvaluator`エージェントを作成します。


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## まとめ

このレッスンでは、Microsoft Agent Framework を使用して **メタ認知エージェント** を構築する方法を学びました:

- **自己反省**: 自分自身の推論を監視し、何が起きたかを透明に伝えるエージェント。
- **フォールバックによるエラー回復**: エージェントが障害（例: 404 エラー）を検出すると、自動的に代替ソースを試す、プライマリ＋バックアップのツールパターン。
- **自己評価**: 応答の完全性、正確性、および有用性を評価してスコアをつける、別の評価エージェント。

これらのパターンはエージェントをより堅牢で透明性が高く、信頼できるものにします — 本番環境へのデプロイで重要な特性です。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
免責事項：
この文書はAI翻訳サービス「Co-op Translator」（https://github.com/Azure/co-op-translator）を用いて翻訳されました。弊社は正確性に努めていますが、自動翻訳には誤りや不正確な表現が含まれる場合があります。原文（原言語による文書）を正式な情報源とみなしてください。重要な情報については、専門の人による翻訳を推奨します。本翻訳の利用により生じたいかなる誤解や解釈の相違についても、弊社は責任を負いません。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
